In [51]:
# statistical_analysis.py
# Simple stats workflow:
# 1) z-scores within seed + boxplots
# 2) Friedman ANOVA (>=3 configs)
# 3) Wilcoxon signed-rank post-hoc (pairwise between configs, Bonferroni)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ----------------------------
# 1) Z-scores + boxplots
# # ----------------------------
# def zscores_and_boxplots(df, seed_col, config_col, configs, measures, make_plots=True):
#     """
#     - Filters df to selected configs
#     - Keeps only seeds present in all configs (paired)
#     - Computes z-scores per seed (across configs) for each measure
#     - Optionally plots boxplots per measure (one plot per measure)
#     Returns: d (filtered df with <measure>_z columns), wide_tables (dict measure->wide z table)
#     """
#     d = df[df[config_col].isin(configs)].copy()
#     d[seed_col] = d[seed_col].astype(str)
# 
#     common_seeds = set.intersection(*[set(d.loc[d[config_col] == c, seed_col]) for c in configs])
#     d = d[d[seed_col].isin(common_seeds)].copy()
# 
#     for m in measures:
#         d[m] = pd.to_numeric(d[m], errors="coerce")
#         mu = d.groupby(seed_col)[m].transform("mean")
#         sd = d.groupby(seed_col)[m].transform(lambda x: x.std(ddof=0))
#         d[m + "_z"] = (d[m] - mu) / sd.replace(0, np.nan)
# 
#     wide_tables = {}
#     for m in measures:
#         zc = m + "_z"
#         wide = d.pivot_table(index=seed_col, columns=config_col, values=zc, aggfunc="mean")
#         wide = wide[configs].dropna(axis=0, how="any")
#         wide_tables[m] = wide
# 
#         if make_plots:
#             groups = [wide[c].to_numpy() for c in wide.columns]
#             plt.figure(figsize=(12, 4))
#             plt.boxplot(groups, labels=list(wide.columns), showmeans=True)
#             plt.axhline(0.0, linestyle="--")
#             plt.title(f"{m} (z-scores within seed)")
#             plt.ylabel("z-score [-]")
#             plt.xticks(rotation=35, ha="right")
#             plt.tight_layout()
#             plt.show()
# 
#     return d, wide_tables
# 
# 
# # ----------------------------
# # 2) Friedman ANOVA
# # ----------------------------
# def friedman_anova(df, seed_col, config_col, configs, measures, use_zscores=True):
#     """
#     - Filters df to selected configs
#     - Keeps only paired seeds across configs
#     - Runs Friedman per measure
#     Returns: friedman_results (DataFrame), wide_tables (dict measure->wide table used)
#     """
#     d = df[df[config_col].isin(configs)].copy()
#     d[seed_col] = d[seed_col].astype(str)
# 
#     common_seeds = set.intersection(*[set(d.loc[d[config_col] == c, seed_col]) for c in configs])
#     d = d[d[seed_col].isin(common_seeds)].copy()
# 
#     if use_zscores:
#         for m in measures:
#             d[m] = pd.to_numeric(d[m], errors="coerce")
#             mu = d.groupby(seed_col)[m].transform("mean")
#             sd = d.groupby(seed_col)[m].transform(lambda x: x.std(ddof=0))
#             d[m + "_z"] = (d[m] - mu) / sd.replace(0, np.nan)
# 
#     rows = []
#     wide_tables = {}
# 
#     for m in measures:
#         col = (m + "_z") if use_zscores else m
#         wide = d.pivot_table(index=seed_col, columns=config_col, values=col, aggfunc="mean")
#         wide = wide[configs].dropna(axis=0, how="any")
#         wide_tables[m] = wide
# 
#         arrays = [wide[c].to_numpy() for c in wide.columns]
#         chi2, p = stats.friedmanchisquare(*arrays)
# 
#         n, k = wide.shape
#         W = chi2 / (n * (k - 1))  # Kendall's W
#         df_chi2 = k - 1
# 
#         rows.append({
#             "measure": m,
#             "chi2": float(chi2),
#             "df": int(df_chi2),
#             "p": float(p),
#             "n": int(n),
#             "k": int(k),
#             "kendall_W": float(W),
#         })
# 
#     friedman_results = pd.DataFrame(rows).sort_values("p")
#     return friedman_results, wide_tables
# 
# 
# # ----------------------------
# # 3) Wilcoxon post-hoc (pairwise)
# # ----------------------------
# def wilcoxon_posthoc(df, seed_col, config_col, configs, measures, use_zscores=True, correction="bonferroni"):
#     """
#     - Filters df to selected configs
#     - Keeps only paired seeds across configs
#     - Runs pairwise Wilcoxon signed-rank per measure
#     - Bonferroni by default: p_adj = p_raw * (#pairs)
#     Returns: posthoc_results (DataFrame)
#     """
#     d = df[df[config_col].isin(configs)].copy()
#     d[seed_col] = d[seed_col].astype(str)
# 
#     common_seeds = set.intersection(*[set(d.loc[d[config_col] == c, seed_col]) for c in configs])
#     d = d[d[seed_col].isin(common_seeds)].copy()
# 
#     if use_zscores:
#         for m in measures:
#             d[m] = pd.to_numeric(d[m], errors="coerce")
#             mu = d.groupby(seed_col)[m].transform("mean")
#             sd = d.groupby(seed_col)[m].transform(lambda x: x.std(ddof=0))
#             d[m + "_z"] = (d[m] - mu) / sd.replace(0, np.nan)
# 
#     pairs = [(configs[i], configs[j]) for i in range(len(configs)) for j in range(i + 1, len(configs))]
#     n_pairs = len(pairs)
# 
#     rows = []
#     for m in measures:
#         col = (m + "_z") if use_zscores else m
#         wide = d.pivot_table(index=seed_col, columns=config_col, values=col, aggfunc="mean")
#         wide = wide[configs].dropna(axis=0, how="any")
# 
#         for a, b in pairs:
#             x = wide[a].to_numpy()
#             y = wide[b].to_numpy()
# 
#             stat, p = stats.wilcoxon(x, y)
#             rows.append({
#                 "measure": m,
#                 "A": a,
#                 "B": b,
#                 "stat": float(stat),
#                 "p_raw": float(p),
#                 "n": int(len(x)),
#                 "median_diff": float(np.median(x - y)),
#                 "mean_diff": float(np.mean(x - y)),
#             })
# 
#     posthoc = pd.DataFrame(rows)
# 
#     if correction == "bonferroni":
#         posthoc["p_adj"] = posthoc["p_raw"] * n_pairs
#     else:
#         posthoc["p_adj"] = posthoc["p_raw"]
# 
#     return posthoc.sort_values(["measure", "p_adj", "p_raw"])

In [52]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)          # 0 = auto, gebruik volledige notebook-breedte
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)
# =========================================================
# Core idea
# You compare CONDITIONS (e.g. scenarios) while BLOCKING on
# the UNIT (e.g. seed). So each seed is one paired row:
#
#   seed_i : condition A value, condition B value, ...
#
# Everything below is just building that paired wide table.
# =========================================================


#inputs: df
# unit col: unit om mee te pairen, seed of scenario
# condition_col: kolom om te vergelijken, in dit geval config
# conditions: lijst met conditions uit conditions_col
# measure: KPI's
# z score: z score meten of raw values meten
def make_paired_wide(df, unit_col, condition_col, conditions, measure, zscore_within_unit=False):
    """
    Build a paired wide table for ONE measure.

    Returns: wide DataFrame
        index   = unit (e.g. seed)
        columns = condition (e.g. scenario), in the given order
        values  = mean per (unit, condition) if duplicates exist

    Steps:
      1) filter to conditions
      2) keep only units present in ALL conditions (paired)
      3) coerce measure to numeric
      4) optional: z-score within each unit across conditions
      5) pivot to wide + drop incomplete rows
    """
    # filter conditions
    d = df[df[condition_col].isin(conditions)].copy()

    # 1) Ensure unit is comparable (seeds sometimes int/str mixed)
    d[unit_col] = d[unit_col].astype(str)

    # 2) Keep only paired units (intersection)
    common_units = set.intersection(*[
        set(d.loc[d[condition_col] == c, unit_col]) for c in conditions
    ])
    d = d[d[unit_col].isin(common_units)].copy()

    # 3) Numeric measure
    d[measure] = pd.to_numeric(d[measure], errors="coerce")

    # 4) Optional z-score within unit (across conditions)
    value_col = measure
    if zscore_within_unit:
        mu = d.groupby(unit_col)[measure].transform("mean")
        sd = d.groupby(unit_col)[measure].transform(lambda x: x.std(ddof=0)).replace(0, np.nan)
        d[measure + "_z"] = (d[measure] - mu) / sd
        value_col = measure + "_z"

    # 5) Wide table seed × condition (mean if duplicates)
    wide = d.pivot_table(
        index=unit_col,
        columns=condition_col,
        values=value_col,
        aggfunc="mean",
    )

    # Enforce column order + complete cases
    wide = wide[conditions].dropna(axis=0, how="any")
    return wide


# ----------------------------
# 1) Z-scores + boxplots
# ----------------------------
def zscores_and_boxplots(df, seed_col, config_col, configs, measures, make_plots=True):
    """
    Returns:
      d_z: long df filtered to paired seeds with <measure>_z columns
      wide_tables: dict {measure: wide_z_table}
    """
    d = df[df[config_col].isin(configs)].copy()
    d[seed_col] = d[seed_col].astype(str)

    common_units = set.intersection(*[
        set(d.loc[d[config_col] == c, seed_col]) for c in configs
    ])
    d = d[d[seed_col].isin(common_units)].copy()

    # numeric + zscores within seed
    for m in measures:
        d[m] = pd.to_numeric(d[m], errors="coerce")
        mu = d.groupby(seed_col)[m].transform("mean")
        sd = d.groupby(seed_col)[m].transform(lambda x: x.std(ddof=0)).replace(0, np.nan)
        d[m + "_z"] = (d[m] - mu) / sd

    wide_tables = {}
    for m in measures:
        wide = d.pivot_table(index=seed_col, columns=config_col, values=m + "_z", aggfunc="mean")
        wide = wide[configs].dropna(axis=0, how="any")
        wide_tables[m] = wide

        if make_plots:
            groups = [wide[c].to_numpy() for c in wide.columns]
            plt.figure(figsize=(12, 4))
            plt.boxplot(groups, labels=list(wide.columns), showmeans=True)
            plt.axhline(0.0, linestyle="--")
            plt.title(f"{m} (z-scores within seed)")
            plt.ylabel("z-score [-]")
            plt.xticks(rotation=35, ha="right")
            plt.tight_layout()
            plt.show()

    return d, wide_tables


# ----------------------------
# 2) Friedman ANOVA
# ----------------------------
def friedman_anova(df, seed_col, config_col, configs, measures, use_zscores=True, alpha=0.05):
    rows = []
    wide_tables = {}

    for m in measures:
        wide = make_paired_wide(
            df, unit_col=seed_col, condition_col=config_col,
            conditions=configs, measure=m, zscore_within_unit=use_zscores
        )
        wide_tables[m] = wide

        arrays = [wide[c].to_numpy() for c in wide.columns]
        chi2, p = stats.friedmanchisquare(*arrays)

        n, k = wide.shape
        W = chi2 / (n * (k - 1))

        rows.append({
            "measure": m,
            "chi2": float(chi2),
            "df": int(k - 1),
            "p": float(p),
            "alpha": float(alpha),
            "significant": bool(p <= alpha),
            "n": int(n),
            "k": int(k),
            "kendall_W": float(W),
            "used_zscores": bool(use_zscores),
        })

    return pd.DataFrame(rows).sort_values("p"), wide_tables

# ----------------------------
# 3) Wilcoxon post-hoc (pairwise)
# ----------------------------
def wilcoxon_posthoc(df, seed_col, config_col, configs, measures,
                     use_zscores=True, correction="bonferroni", alpha=0.05):
    pairs = [(configs[i], configs[j]) for i in range(len(configs)) for j in range(i + 1, len(configs))]
    n_pairs = len(pairs)

    rows = []
    for m in measures:
        wide = make_paired_wide(
            df, unit_col=seed_col, condition_col=config_col,
            conditions=configs, measure=m, zscore_within_unit=use_zscores
        )

        for a, b in pairs:
            x = wide[a].to_numpy()
            y = wide[b].to_numpy()
            diff = x - y

            if np.all(np.isfinite(diff)) and np.allclose(diff, 0.0):
                stat, p = 0.0, 1.0
            else:
                try:
                    stat, p = stats.wilcoxon(x, y)
                except Exception:
                    stat, p = np.nan, np.nan

            rows.append({
                "measure": m,
                "A": a,
                "B": b,
                "stat": float(stat) if stat is not None else np.nan,
                "p_raw": float(p) if p is not None else np.nan,
                "n": int(len(x)),
                "median_diff": float(np.nanmedian(diff)),
                "mean_diff": float(np.nanmean(diff)),
                "used_zscores": bool(use_zscores),
            })

    posthoc = pd.DataFrame(rows)

    if correction == "bonferroni":
        posthoc["p_adj"] = np.minimum(posthoc["p_raw"] * n_pairs, 1.0)
    else:
        posthoc["p_adj"] = posthoc["p_raw"]

    posthoc["alpha"] = float(alpha)
    posthoc["significant"] = posthoc["p_adj"] <= alpha

    return posthoc.sort_values(["measure", "p_adj", "p_raw"])

In [53]:
# === Cell 2: load ALL pickles from folder -> single DataFrame ===
from pathlib import Path
import pickle

PICKLE_DIR = Path("/Users/jornvanbeek/Desktop/bluesky/Montecarlo")  # <-- map met pickles
# PICKLE_DIR = Path('/Users/jornvanbeek/Desktop/bluesky/Montecarlo/old_errorgenerator')
DF_KEY = None  # bv "results_df" als elke pickle een dict is

def load_pickle(path: Path):
    with path.open("rb") as f:
        return pickle.load(f)

dfs = []

pickle_files = sorted(PICKLE_DIR.glob("*.pkl")) + sorted(PICKLE_DIR.glob("*.pickle"))
if not pickle_files:
    raise FileNotFoundError(f"No pickle files found in {PICKLE_DIR}")

for pkl in pickle_files:
    obj = load_pickle(pkl)

    # --- haal DataFrame uit pickle ---
    if isinstance(obj, pd.DataFrame):
        df_i = obj.copy()

    elif isinstance(obj, dict):
        if DF_KEY is not None:
            if DF_KEY not in obj or not isinstance(obj[DF_KEY], pd.DataFrame):
                raise KeyError(f"{pkl.name}: DF_KEY={DF_KEY!r} not found")
            df_i = obj[DF_KEY].copy()
        else:
            for k in ("results_df", "results", "df", "data"):
                if k in obj and isinstance(obj[k], pd.DataFrame):
                    df_i = obj[k].copy()
                    break
            else:
                raise KeyError(f"{pkl.name}: no DataFrame key found")

    else:
        raise TypeError(f"{pkl.name}: unsupported pickle content {type(obj)}")

    # --- voeg configuratie-label toe ---
    # standaard: bestandsnaam zonder extensie
    df_i["config"] = pkl.stem

    dfs.append(df_i)

# combineer alles
df = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(dfs)} pickles")
print("Combined shape:", df.shape)
print("Configs:", df["config"].unique())


Loaded 24 pickles
Combined shape: (4428, 92)
Configs: ['eaman_BOL' 'eaman_BOL_25' 'eaman_BOL_25_higherseed'
 'eaman_BOL_25_seed32' 'eaman_BOL_higherseed' 'eaman_BOL_seed32'
 'eaman_delay' 'eaman_delay_25' 'eaman_delay_25_higherseed'
 'eaman_delay_higherseed' 'eaman_delay_seed32' 'eaman_fcfs'
 'eaman_fcfs_25' 'eaman_fcfs_25_higherseed' 'eaman_fcfs_25_seed32'
 'eaman_fcfs_higherseed' 'eaman_fcfs_seed32' 'eaman_no_popup'
 'eaman_no_popup_higherseed' 'eaman_no_popup_seed32'
 'eaman_zero_uncertainty' 'standard_aman' 'standard_aman_higherseed'
 'standard_aman_seed32']


In [54]:
df['configuration'].value_counts()

configuration
BOL20            576
BOL25            576
delay20          576
FCFS20           576
FCFS25           576
FCFS20nopopup    576
FCFS14           576
delay25          384
FCFS20certain     12
Name: count, dtype: int64

In [55]:
df

,scenario,run,seed,usecache,node,status,maxtime,starttime,endtime,elapsed,configuration,cachename,pct_extrawork,freezehorizon,popup_planner,error_multiplicator,capacity,pct_count_eq_0,mean_count,mean_count_nonzero,max_count,second_highest_count,max_count_acid,min_count_holding,mean_count_holding,max_count_holding,count_popup,count_fh_margin_at_freeze,min_fh_margin_at_freeze,mean_fh_margin_at_freeze,max_fh_margin_at_freeze,min_EAT_updates,max_EAT_updates,total_EAT_updates,min_swaps,mean_swaps,max_swaps,mean_swaps_nonzero,amount_of_swaps,min_ttlg_at_freeze,mean_ttlg_at_freeze,median_ttlg_at_freeze,max_ttlg_at_freeze,mean_abs_ttlg_at_freeze,mean_E_TO,mean_abs_E_TO,min_E_TO,max_E_TO,mean_totaldelay,max_totaldelay,mean_totaldelay_nonzero,min_totalspeedup,mean_totalspeedup,mean_totalspeedup_nonzero,mean_LLDA,max_LLDA,mean_LLDA_nonzero,count_LLDA_nonzero,count_flights_with_mach_instr,count_flights_with_adjacent_instr,count_flights_with_mach_or_adj_instr,min_short_adjacent,mean_short_adjacent,mean_short_adjacent_nonzero,count_short_adjacent_nonzero,min_delay_mach,mean_delay_mach,mean_delay_mach_nonzero,count_delay_mach_nonzero,mean_short_speed,mean_delay_speed,mean_delay_dogleg,mean_short_dogleg,mean_abs_eat_adherence,max_abs_eat_adherence,mean_TP_accuracy,max_abs_TP_accuracy,mean_time_error_at_freeze,max_time_error_at_freeze,min_time_error_at_freeze,mean_abs_time_error_at_freeze,pct_extrawork_popup,pct_extrawork_nonpopup,mean_minwork,mean_totalwork,mean_extrawork,mean_percentile_time,mean_slot_minus_initialslot,mean_abs_slot_minus_initialslot_nonzero,n_acids,error_seed,config
0,sc1,0,0,True,b'SG\xa2#\xcd',done,07:00:00,2026-02-09 23:55:31.542410,2026-02-10 00:03:09.890603,458.348193,BOL20,sc1,-0.100055,1200.0,BACK,"[1.0, 1.0, 1.0, 1.0]",38.0,58.823529,0.661765,1.607143,5.0,3.0,KLM90X,NaN,NaN,NaN,0,0,NaN,NaN,NaN,0.0,1.0,14.0,0.0,0.00495,1.0,1.0,1.0,-60.005195,70.462073,53.531348,465.095472,98.533793,0.015982,0.025541,-0.975044,4.235333,100.803419,421.0,143.829268,-81.0,-0.991453,-29.0,9.073529,249.0,154.25,12,16,1,17,-10.0,-0.049505,-10.0,1,0.0,5.407895,25.6875,16,-1.152174,30.207317,18.51,NaN,52.381863,120.9,9.514706,206.0,1.511698,95.663119,-86.314019,20.989078,NaN,NaN,37487335713.985916,37449865161.426895,-37470552.559024,2.757601,-1.113562,58.971753,204,2443250962,eaman_BOL
1,sc1,1,1,True,b'SG\xa2#\xcf',done,07:00:00,2026-02-09 23:55:31.592544,2026-02-10 00:02:50.237750,438.645206,BOL20,sc1,0.17775,1200.0,BACK,"[1.0, 1.0, 1.0, 1.0]",38.0,69.117647,0.607843,1.968254,5.0,3.0,KLM42TSH,NaN,NaN,NaN,0,0,NaN,NaN,NaN,0.0,1.0,4.0,0.0,0.009804,2.0,2.0,2.0,-60.002232,56.422119,45.863473,369.327945,86.453089,0.065636,0.065636,0.0,12.428961,97.380435,385.0,149.316667,-42.0,-1.173913,-27.0,9.838235,183.0,133.8,15,16,1,17,-39.0,-0.191176,-39.0,1,0.0,5.532468,26.625,16,-0.676471,14.571429,22.3,NaN,51.077941,116.8,7.622549,167.0,3.059217,190.895283,-87.35322,20.198369,NaN,NaN,37487335713.985916,37554088129.681602,66752415.695688,2.757601,-0.959301,159.006707,204,2629073562,eaman_BOL
2,sc1,2,2,True,b'SG\xa2#\xc3',done,07:00:00,2026-02-09 23:55:31.642552,2026-02-10 00:02:53.752810,442.110258,BOL20,sc1,0.061989,1200.0,BACK,"[1.0, 1.0, 1.0, 1.0]",38.0,65.686275,0.470588,1.371429,8.0,5.0,KLM28H,8.0,8.0,8.0,0,0,NaN,NaN,NaN,0.0,1.0,14.0,0.0,0.010204,1.0,1.0,2.0,-60.000127,45.350801,26.788432,428.442078,77.391689,0.033392,0.033392,0.0,6.260019,78.533358,441.269056,122.163001,-138.0,-5.540816,-67.875,3.735294,192.0,152.4,5,5,5,10,-122.0,-8.413043,-77.4,5,0.0,1.484211,28.2,5,-0.893617,18.492063,9.180723,NaN,52.883333,117.3,9.612745,237.0,3.24742,107.740591,-126.920469,19.985746,NaN,NaN,37487335713.985916,37510588305.04966,23252591.063745,2.757601,-1.216021,48.145495,204,1091773355,eaman_BOL
3,sc1,3,3,True,b'SG\xa2#\xc8',done,07:00:00,2026-02-09 23:55:31.692446,2026-02-10 00:03:08.994909,457.302463,BOL20,sc1,0.012011,1200.0,BACK,"[1.0, 1.0, 1.0, 1.0]",38.0,65.686275,0.607843,1.771429,6.0,5.0,AAL204SH,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.

In [59]:
# %% =========================
# GLOBAL: choose measures once
# ===========================

# Default measure sets (edit here; experiment cells typically just pick one of these)
MEASURES_DEFAULT = [
    "mean_LLDA",
    "count_LLDA_nonzero",
    "pct_extrawork",          # was mean_pct_extrawork before
    "total_EAT_updates",
    "amount_of_swaps",
    "mean_totaldelay",
]

# MEASURES_WORKLOAD = [
#     "pct_extrawork",
#     "total_EAT_updates",
#     "amount_of_swaps",
# ]
# 
# MEASURES_DELAY = [
#     "LLDA",
#     "mean_totaldelay",
# ]

# Select which set to use by default for all experiments
MEASURES = MEASURES_DEFAULT

# Pairing / grouping columns (edit once)
SEED_COL = "scenario"       # paired unit
COND_COL = "configuration"   # condition column you compare (scenario OR config OR title etc.)

# Common knobs
USE_ZSCORES = True
MAKE_PLOTS = True
P_CORR = "bonferroni"

In [57]:
# %% ==========================================
# EXP 1 — effect of uncertainty (compare configs)
# =============================================

# Set which column defines the compared conditions (only change if needed)
seed_col = SEED_COL
config_col = COND_COL

# Conditions to compare (edit these strings to your actual labels)
configs = [
    "FCFS20certain",          # example: zero uncertainty / deterministic
    "FCFS20nopopup",
    "FCFS20"
    # example: uncertainty on
    # add more if you want (Friedman needs >=3; Wilcoxon works pairwise anyway)
]

measures = MEASURES  # only change if there is a reason: e.g. measures = MEASURES_DELAY

# Run
d_z, wide_z = zscores_and_boxplots(df, seed_col, config_col, configs, measures, make_plots=MAKE_PLOTS)
friedman_results, _ = friedman_anova(df, seed_col, config_col, configs, measures, use_zscores=USE_ZSCORES)
posthoc = wilcoxon_posthoc(df, seed_col, config_col, configs, measures, use_zscores=USE_ZSCORES, correction=P_CORR)

print("Friedman:\n", friedman_results)
print("\nPost-hoc Wilcoxon:\n", posthoc)

KeyError: "None of [Index(['FCFS20certain', 'FCFS20nopopup', 'FCFS20'], dtype='object', name='config')] are in the [columns]"

In [ ]:
wide_z

In [ ]:
# %% ==================================================
# EXP 2 — effect of horizon extension (compare horizons)
# =====================================================

seed_col = SEED_COL
config_col = COND_COL

configs = [
    "FCFS14",              # example: baseline horizon
    "FCFS20",              # example: extended horizon
    "FCFS25",              # example: extra extended horizon (optional)
]

measures = MEASURES  # often keep default; delay/stability measures are most relevant here

d_z, wide_z = zscores_and_boxplots(df, seed_col, config_col, configs, measures, make_plots=MAKE_PLOTS)
friedman_results, _ = friedman_anova(df, seed_col, config_col, configs, measures, use_zscores=USE_ZSCORES)
posthoc = wilcoxon_posthoc(df, seed_col, config_col, configs, measures, use_zscores=USE_ZSCORES, correction=P_CORR)

print("Friedman:\n", friedman_results)
print("\nPost-hoc Wilcoxon:\n", posthoc)

In [ ]:
# %% ===================================================
# EXP 3 — different schedulers (same horizon, compare)
# =====================================================

seed_col = SEED_COL
config_col = COND_COL

configs = [
    "delay20",             # example labels
    "FCFS20",
    "BOL20",
]

measures = MEASURES  # scheduler comparison often: workload/stability; change if you want

d_z, wide_z = zscores_and_boxplots(df, seed_col, config_col, configs, measures, make_plots=MAKE_PLOTS)
friedman_results, _ = friedman_anova(df, seed_col, config_col, configs, measures, use_zscores=USE_ZSCORES)
posthoc = wilcoxon_posthoc(df, seed_col, config_col, configs, measures, use_zscores=USE_ZSCORES, correction=P_CORR)

print("Friedman:\n", friedman_results)
print("\nPost-hoc Wilcoxon:\n", posthoc)

In [ ]:
wide_z

In [ ]:
# %% ==================================================================
# EXP 4 — different schedulers with extra-extended horizon (compare)
# ====================================================================

seed_col = SEED_COL
config_col = COND_COL

configs = [
    "delay25",        # example: same schedulers but now with extra-extended FH
    "FCFS25",
    "BOL25",
]

measures = MEASURES  # here you often want both delay + workload; default set is fine

d_z, wide_z = zscores_and_boxplots(df, seed_col, config_col, configs, measures, make_plots=MAKE_PLOTS)
friedman_results, _ = friedman_anova(df, seed_col, config_col, configs, measures, use_zscores=USE_ZSCORES)
posthoc = wilcoxon_posthoc(df, seed_col, config_col, configs, measures, use_zscores=USE_ZSCORES, correction=P_CORR)

print("Friedman:\n", friedman_results)
print("\nPost-hoc Wilcoxon:\n", posthoc)

In [ ]:
df

In [58]:
# %% ==================================================================
# EXP 4 — different schedulers with extra-extended horizon (compare)
# ====================================================================

seed_col = SEED_COL
config_col = COND_COL

configs = [
    "FCFS14",        # example: same schedulers but now with extra-extended FH
    "delay20",
    "BOL20",
]

measures = MEASURES  # here you often want both delay + workload; default set is fine

d_z, wide_z = zscores_and_boxplots(df, seed_col, config_col, configs, measures, make_plots=MAKE_PLOTS)
friedman_results, _ = friedman_anova(df, seed_col, config_col, configs, measures, use_zscores=USE_ZSCORES)
posthoc = wilcoxon_posthoc(df, seed_col, config_col, configs, measures, use_zscores=USE_ZSCORES, correction=P_CORR)

print("Friedman:\n", friedman_results)
print("\nPost-hoc Wilcoxon:\n", posthoc)

KeyError: "None of [Index(['FCFS14', 'delay20', 'BOL20'], dtype='object', name='config')] are in the [columns]"